In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
from datetime import datetime, timedelta
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 14:15:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
#A. Menyiapkan Streaming Source
# Menyiapkan folder simulasi (dan membersihkan sisa percobaan sebelumnya)
folder_stream = "tugas10_stream"
shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

# Skema eksplisit data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# Input source streaming
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("df_stream.isStreaming:", df_stream.isStreaming)

df_stream.isStreaming: True


In [3]:
#B. Agregasi per Cabang
# Kolom turunan total_harga
df_pesanan = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per cabang
agregasi_cabang_stream = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_penjualan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Streaming query dengan sink memory
query_cabang = agregasi_cabang_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang dimulai. Aktif:", query_cabang.isActive)

Query agregasi_cabang dimulai. Aktif: True


In [4]:
#C. Agregasi Berbasis Window Waktu
# readStream KEDUA dari folder yang sama
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

df_pesanan_waktu = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per window 2 menit
agregasi_waktu_stream = df_pesanan_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_penjualan")
)

query_waktu = agregasi_waktu_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu dimulai. Aktif:", query_waktu.isActive)

Query agregasi_waktu dimulai. Aktif: True


In [6]:
#D. Menguji dengan Data Simulasi
# Batch 1
waktu_sekarang = datetime.now()

pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Latte"],
    "qty": [2, 1, 3],
    "harga_satuan": [25000, 20000, 28000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 3,
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(8)

print("=== agregasi_cabang setelah Batch 1 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch 1 dikirim. Menunggu Spark memproses...
=== agregasi_cabang setelah Batch 1 ===
+-----------+---------------+--------------+
|nama_cabang|total_penjualan|jumlah_pesanan|
+-----------+---------------+--------------+
|   Magelang|         134000|             2|
|       Solo|          20000|             1|
+-----------+---------------+--------------+



In [7]:
# Batch 2
waktu_sekarang = datetime.now()

pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Yogyakarta", "Solo", "Magelang"],
    "nama_menu": ["Kopi Susu", "Latte", "Americano"],
    "qty": [4, 2, 1],
    "harga_satuan": [25000, 28000, 20000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 3,
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(8)

print("=== agregasi_cabang setelah Batch 2 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

Batch 2 dikirim. Menunggu Spark memproses...


=== agregasi_cabang setelah Batch 2 ===
+-----------+---------------+--------------+
|nama_cabang|total_penjualan|jumlah_pesanan|
+-----------+---------------+--------------+
|   Magelang|         154000|             3|
|       Solo|          76000|             2|
| Yogyakarta|         100000|             1|
+-----------+---------------+--------------+



In [8]:
# Batch 3
# Waktu 5 menit ke depan agar jatuh di window yang berbeda
waktu_berbeda = datetime.now() + timedelta(minutes=5)

pd.DataFrame({
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Yogyakarta", "Semarang", "Solo"],
    "nama_menu": ["Latte", "Kopi Susu", "Americano"],
    "qty": [2, 3, 2],
    "harga_satuan": [28000, 25000, 20000],
    "waktu_pesan": [waktu_berbeda.strftime("%Y-%m-%d %H:%M:%S")] * 3,
}).to_csv(f"{folder_stream}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(8)

print("=== agregasi_cabang setelah Batch 3 ===")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

print("=== agregasi_waktu (total per window 2 menit) ===")
spark.sql("""
    SELECT window.start, window.end, total_penjualan
    FROM agregasi_waktu
    ORDER BY window.start
""").show(truncate=False)

Batch 3 dikirim. Menunggu Spark memproses...


=== agregasi_cabang setelah Batch 3 ===
+-----------+---------------+--------------+
|nama_cabang|total_penjualan|jumlah_pesanan|
+-----------+---------------+--------------+
|   Magelang|         154000|             3|
|   Semarang|          75000|             1|
|       Solo|         116000|             3|
| Yogyakarta|         156000|             2|
+-----------+---------------+--------------+

=== agregasi_waktu (total per window 2 menit) ===
+-------------------+-------------------+---------------+
|start              |end                |total_penjualan|
+-------------------+-------------------+---------------+
|2026-11-05 14:18:00|2026-11-05 14:20:00|154000         |
|2026-11-05 14:20:00|2026-11-05 14:22:00|176000         |
|2026-11-05 14:26:00|2026-11-05 14:28:00|171000         |
+-------------------+-------------------+---------------+



In [9]:
#E. Sink Berbasis Berkas (Tantangan)
# Streaming query ketiga: data mentah ke Parquet
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query Parquet dimulai. Aktif:", query3.isActive)

Query Parquet dimulai. Aktif: True


In [11]:
waktu_sekarang = datetime.now()

pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Latte", "Kopi Susu"],
    "qty": [1, 2],
    "harga_satuan": [28000, 25000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2,
}).to_csv(f"{folder_stream}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark menulis Parquet...")
time.sleep(10)

Batch 4 dikirim. Menunggu Spark menulis Parquet...


In [12]:
# Bukti 1: berkas Parquet benar-benar tersimpan
!ls tugas10_output

part-00000-27e2dc53-434c-4cd0-8a6a-b0b762e69a9f-c000.snappy.parquet
part-00000-f75b32a0-b46b-41ad-9638-3531cce2a8aa-c000.snappy.parquet
part-00001-3a6f1a1f-b88b-4c3b-9a2f-7c4b9fc3cfb7-c000.snappy.parquet
_spark_metadata


In [13]:
# Bukti 2: dibaca kembali sebagai DataFrame BATCH biasa
df_hasil = spark.read.parquet("tugas10_output")
df_hasil.orderBy("pesanan_id").show()
print("Jumlah baris:", df_hasil.count())

+----------+-----------+---------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+---------+---+------------+-------------------+
|      P001|   Magelang|Kopi Susu|  2|       25000|2026-11-05 14:18:35|
|      P002|       Solo|Americano|  1|       20000|2026-11-05 14:18:35|
|      P003|   Magelang|    Latte|  3|       28000|2026-11-05 14:18:35|
|      P004| Yogyakarta|Kopi Susu|  4|       25000|2026-11-05 14:20:02|
|      P005|       Solo|    Latte|  2|       28000|2026-11-05 14:20:02|
|      P006|   Magelang|Americano|  1|       20000|2026-11-05 14:20:02|
|      P007| Yogyakarta|    Latte|  2|       28000|2026-11-05 14:26:22|
|      P008|   Semarang|Kopi Susu|  3|       25000|2026-11-05 14:26:22|
|      P009|       Solo|Americano|  2|       20000|2026-11-05 14:26:22|
|      P010|   Magelang|    Latte|  1|       28000|2026-11-05 14:23:49|
|      P011|       Solo|Kopi Susu|  2|       25000|2026-11-05 14

In [14]:
# Tambahan
df_rate = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

df_rate.printSchema()

q_rate = df_rate.writeStream \
    .format("memory") \
    .queryName("tabel_rate") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(8)
spark.sql("SELECT COUNT(*) AS jumlah_baris FROM tabel_rate").show()
spark.sql("SELECT * FROM tabel_rate ORDER BY value DESC LIMIT 5").show(truncate=False)

q_rate.stop()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)

+------------+
|jumlah_baris|
+------------+
|          30|
+------------+

+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-11-05 14:39:53.283|29   |
|2026-11-05 14:39:53.083|28   |
|2026-11-05 14:39:52.883|27   |
|2026-11-05 14:39:52.683|26   |
|2026-11-05 14:39:52.483|25   |
+-----------------------+-----+



In [25]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os, shutil, time
import pandas as pd

skema_waktu = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
    StructField("waktu_transaksi", TimestampType(), True),
])

def siapkan_folder(nama):
    shutil.rmtree(nama, ignore_errors=True)
    os.makedirs(nama, exist_ok=True)
    return nama

def hentikan(nama_query):
    for q in spark.streams.active:
        if q.name == nama_query:
            q.stop()

def kirim_csv(folder, nama_file, baris, kolom, jeda=10):
    pd.DataFrame(baris, columns=kolom).to_csv(f"{folder}/{nama_file}", index=False)
    time.sleep(jeda)

In [26]:
hentikan("tabel_rate")

df_rate = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

q_rate = df_rate.writeStream \
    .format("memory") \
    .queryName("tabel_rate") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(10)

spark.sql("SELECT COUNT(*) AS jumlah_baris FROM tabel_rate").show()
spark.sql("SELECT * FROM tabel_rate ORDER BY value DESC LIMIT 3").show(truncate=False)

p = q_rate.lastProgress
if p:
    print("Batch ke-", p["batchId"], "| baris masuk:", p["numInputRows"])

q_rate.stop()

+------------+
|jumlah_baris|
+------------+
|          45|
+------------+

+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-11-05 14:53:26.939|44   |
|2026-11-05 14:53:26.739|43   |
|2026-11-05 14:53:26.539|42   |
+-----------------------+-----+

Batch ke- 5 | baris masuk: 10


In [28]:
hentikan("hasil_wm")

folder_wm = siapkan_folder("simulasi_stream_watermark")
kolom = ["trx_id", "kota", "pendapatan", "waktu_transaksi"]

df_wm = spark.readStream.format("csv").schema(skema_waktu) \
    .option("header", True).load(folder_wm)

agregasi_wm = df_wm \
    .withWatermark("waktu_transaksi", "2 minutes") \
    .groupBy(window(col("waktu_transaksi"), "1 minute")) \
    .agg(spark_sum("pendapatan").alias("total_pendapatan"))

q_wm = agregasi_wm.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("hasil_wm") \
    .start()

kueri = "SELECT window.start, window.end, total_pendapatan FROM hasil_wm ORDER BY window.start"

# Batch 1: data menit 10:00 dan 10:01
kirim_csv(folder_wm, "b1.csv", [
    ("W1", "Solo", 100000, "2026-11-05 10:00:10"),
    ("W2", "Solo", 200000, "2026-11-05 10:00:40"),
    ("W3", "Solo", 150000, "2026-11-05 10:01:20"),
], kolom)
print("Setelah batch 1 (belum ada window final, tabel kosong):")
spark.sql(kueri).show(truncate=False)

# Batch 2 dan 3: waktu melompat maju sehingga watermark naik
kirim_csv(folder_wm, "b2.csv", [("W4", "Solo", 300000, "2026-11-05 10:05:00")], kolom)
kirim_csv(folder_wm, "b3.csv", [("W5", "Solo", 50000, "2026-11-05 10:06:00")], kolom)
print("Setelah batch 2-3 (window lama sudah final):")
spark.sql(kueri).show(truncate=False)

# Batch 4: data TERLAMBAT (waktu 10:00 padahal watermark sudah ~10:04)
kirim_csv(folder_wm, "b4.csv", [("W6", "Solo", 999999, "2026-11-05 10:00:30")], kolom)
print("Setelah data terlambat (tidak berubah):")
spark.sql(kueri).show(truncate=False)

q_wm.stop()

Setelah batch 1 (belum ada window final, tabel kosong):
+-----+---+----------------+
|start|end|total_pendapatan|
+-----+---+----------------+
+-----+---+----------------+



Setelah batch 2-3 (window lama sudah final):
+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-05 10:00:00|2026-11-05 10:01:00|300000          |
|2026-11-05 10:01:00|2026-11-05 10:02:00|150000          |
+-------------------+-------------------+----------------+



Setelah data terlambat (tidak berubah):
+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-05 10:00:00|2026-11-05 10:01:00|300000          |
|2026-11-05 10:01:00|2026-11-05 10:02:00|150000          |
+-------------------+-------------------+----------------+



In [30]:
hentikan("hasil_sliding")

folder_sl = siapkan_folder("simulasi_stream_sliding")

df_sl = spark.readStream.format("csv").schema(skema_waktu) \
    .option("header", True).load(folder_sl)

agregasi_sl = df_sl.groupBy(
    window(col("waktu_transaksi"), "10 minutes", "5 minutes")
).agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

q_sl = agregasi_sl.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_sliding") \
    .start()

kirim_csv(folder_sl, "s1.csv", [
    ("S1", "Solo", 100000, "2026-11-05 10:07:00"),
    ("S2", "Solo", 200000, "2026-11-05 10:12:00"),
], ["trx_id", "kota", "pendapatan", "waktu_transaksi"])

spark.sql("""
    SELECT window.start, window.end, total_pendapatan, jumlah_transaksi
    FROM hasil_sliding ORDER BY window.start
""").show(truncate=False)

q_sl.stop()

+-------------------+-------------------+----------------+----------------+
|start              |end                |total_pendapatan|jumlah_transaksi|
+-------------------+-------------------+----------------+----------------+
|2026-11-05 10:00:00|2026-11-05 10:10:00|100000          |1               |
|2026-11-05 10:05:00|2026-11-05 10:15:00|300000          |2               |
|2026-11-05 10:10:00|2026-11-05 10:20:00|200000          |1               |
+-------------------+-------------------+----------------+----------------+



In [29]:
hentikan("hasil_provinsi")

folder_jn = siapkan_folder("simulasi_stream_join")

skema_join = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

df_jn = spark.readStream.format("csv").schema(skema_join) \
    .option("header", True).load(folder_jn)

df_master = spark.createDataFrame(
    [("Magelang", "Jawa Tengah"),
     ("Solo", "Jawa Tengah"),
     ("Yogyakarta", "DI Yogyakarta")],
    ["kota", "provinsi"]
)

df_gabung = df_jn.join(df_master, on="kota", how="inner")

agregasi_prov = df_gabung.groupBy("provinsi").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

q_jn = agregasi_prov.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_provinsi") \
    .start()

kirim_csv(folder_jn, "j1.csv", [
    ("J1", "Magelang", "Makanan", 150000),
    ("J2", "Solo", "Fashion", 200000),
    ("J3", "Yogyakarta", "Elektronik", 500000),
    ("J4", "Semarang", "Makanan", 100000),
], ["trx_id", "kota", "kategori", "pendapatan"])

spark.sql("SELECT * FROM hasil_provinsi ORDER BY provinsi").show()

q_jn.stop()

+-------------+----------------+
|     provinsi|total_pendapatan|
+-------------+----------------+
|DI Yogyakarta|          500000|
|  Jawa Tengah|          350000|
+-------------+----------------+



In [31]:
# Hentikan semua query eksplorasi (berdasarkan nama)
nama_eksplorasi = ["tabel_rate", "hasil_wm", "hasil_sliding", "hasil_provinsi", "fe_query"]

for q in spark.streams.active:
    if q.name in nama_eksplorasi:
        print("Menghentikan:", q.name)
        q.stop()

# Hapus folder simulasi eksplorasi
for folder in ["simulasi_stream_watermark", "simulasi_stream_sliding",
               "simulasi_stream_join", "simulasi_stream_foreach",
               "output_foreach", "checkpoint_foreach"]:
    shutil.rmtree(folder, ignore_errors=True)

print("Query aktif yang tersisa:", [q.name for q in spark.streams.active])
print("Penutup eksplorasi selesai.")

Query aktif yang tersisa: ['agregasi_waktu', 'agregasi_cabang', None]
Penutup eksplorasi selesai.


In [32]:
for q in spark.streams.active:
    q.stop()

spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.
